# Predictor template

1. `encode_batch` reads sequences and builds the model input.
2. `dummy_model` turns that input into coordinates. Replace this function when a real model is ready.
3. `write_pdb` writes one CA atom per residue.

In [1]:
import numpy as np
import torch
from pathlib import Path
from skbio.io import read

In [2]:
DATA_DIR = Path("../data")
OUT_DIR = DATA_DIR / "predictions"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 20 amino acids, X for anything else, PAD for positions added to reach MAX_LENGTH
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY") + ["X", "PAD"]
AA_TO_IDX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
MAX_LENGTH = 1024
BATCH_SIZE = 16

THREE_LETTER = {
    "A": "ALA",
    "C": "CYS",
    "D": "ASP",
    "E": "GLU",
    "F": "PHE",
    "G": "GLY",
    "H": "HIS",
    "I": "ILE",
    "K": "LYS",
    "L": "LEU",
    "M": "MET",
    "N": "ASN",
    "P": "PRO",
    "Q": "GLN",
    "R": "ARG",
    "S": "SER",
    "T": "THR",
    "V": "VAL",
    "W": "TRP",
    "Y": "TYR",
    "X": "UNK",
}

## 1. Input

`read_fasta` uses `skbio.io.read`. `encode_batch` keeps the first `max_length` residues, pads the rest, and returns:

- `tokens` with shape `(B, D, N)` — one-hot, `N = 22`
- `mask` with shape `(B, D)` — `True` for a real residue, `False` for padding
- the letter strings that were actually encoded


In [3]:
def read_fasta(path):
    records = []
    for seq in read(str(path), format="fasta"):
        seq_id = seq.metadata.get("id") or f"sequence_{len(records) + 1}"
        records.append((str(seq_id), str(seq)))
    return records


def encode_batch(records, max_length=MAX_LENGTH):
    standard = set("ACDEFGHIKLMNPQRSTVWY")
    n_aa = len(AMINO_ACIDS)
    tokens = torch.zeros(len(records), max_length, n_aa)
    mask = torch.zeros(len(records), max_length, dtype=torch.bool)
    sequences = []

    for i, (seq_id, sequence) in enumerate(records):
        letters = [aa if aa in standard else "X" for aa in sequence.upper()]
        letters = letters[:max_length]
        sequences.append("".join(letters))

        for j, aa in enumerate(letters):
            tokens[i, j, AA_TO_IDX[aa]] = 1
            mask[i, j] = True
        for j in range(len(letters), max_length):
            tokens[i, j, AA_TO_IDX["PAD"]] = 1

    return tokens, mask, sequences

## 2. Model

Input is the tensor from the encoder. Output is one `(L, 3)` array per protein: X, Y, Z of each CA atom. `L` is the number of `True` values in that row of the mask, so padding is not returned.

This version draws random numbers and ignores the amino-acid channels. A later model has to accept the same two arguments and return the same kind of list.


In [4]:
def dummy_model(tokens, mask):
    batch_size, length, n_aa = tokens.shape
    coords = torch.randn(batch_size, length, 3) * 10

    per_protein = []
    for i in range(batch_size):
        per_protein.append(coords[i][mask[i]])
    return per_protein

## 3. Output

One PDB file per protein, one `ATOM` line per residue. The line is the wwPDB ATOM layout: atom name ` CA ` in columns 13–16, coordinates in columns 31–54, element `C` in columns 77–78.


In [5]:
def write_pdb(sequence, coords, path):
    coords = np.asarray(coords)
    lines = []
    for i, (aa, xyz) in enumerate(zip(sequence, coords), start=1):
        res_name = THREE_LETTER.get(aa, "UNK")
        line = (
            f"ATOM  {i:5d}  CA  {res_name} A{i:4d}    "
            f"{xyz[0]:8.3f}{xyz[1]:8.3f}{xyz[2]:8.3f}  1.00  0.00           C  "
        )
        lines.append(line)
    lines.append("END")
    path.write_text("\n".join(lines) + "\n")

## Run

Change `fasta_path` to any multi-protein FASTA. To swap a stage, replace `encode_batch`, `dummy_model`, or `write_pdb` in the loop below and leave the other two as they are.


In [7]:
DATA_DIR = Path("../data")
SPLIT = "train"
LIMIT = 10

FASTA_DIR = DATA_DIR / "fasta" / SPLIT
REF_PDB_DIR = DATA_DIR / "pdb" / SPLIT
OUT_DIR = DATA_DIR / "predictions" / SPLIT
OUT_DIR.mkdir(parents=True, exist_ok=True)

fasta_files = sorted(FASTA_DIR.glob("*.fasta"))[:LIMIT]
if not fasta_files:
    raise FileNotFoundError(f"No .fasta files in {FASTA_DIR}")

records = []
for fp in fasta_files:
    records.extend(read_fasta(fp))

print(f"Loaded {len(records)} sequences from {FASTA_DIR}:")

torch.manual_seed(0)
eval_pairs = []

for start in range(0, len(records), BATCH_SIZE):
    batch = records[start : start + BATCH_SIZE]
    tokens, mask, sequences = encode_batch(batch)
    print("tokens", tuple(tokens.shape), "lengths", [len(s) for s in sequences])

    with torch.no_grad():
        predictions = dummy_model(tokens, mask)

    for (seq_id, _), sequence, coords in zip(batch, sequences, predictions):
        pred_path = OUT_DIR / f"{seq_id}.pdb"
        ref_path = REF_PDB_DIR / f"{seq_id}.pdb"

        ca_coords = coords[: len(sequence)].detach().cpu().numpy()
        write_pdb(sequence, ca_coords, pred_path)

        if not ref_path.exists():
            raise FileNotFoundError(f"Missing reference file: {ref_path}")

        eval_pairs.append((seq_id, pred_path, ref_path))

eval_pairs

Loaded 10 sequences from ../data/fasta/train:
tokens (10, 1024, 22) lengths [491, 488, 117, 116, 115, 582, 125, 813, 410, 311]


[('AF-A0A067DT54-F1-model_v6',
  PosixPath('../data/predictions/train/AF-A0A067DT54-F1-model_v6.pdb'),
  PosixPath('../data/pdb/train/AF-A0A067DT54-F1-model_v6.pdb')),
 ('AF-A0A067E1K2-F1-model_v6',
  PosixPath('../data/predictions/train/AF-A0A067E1K2-F1-model_v6.pdb'),
  PosixPath('../data/pdb/train/AF-A0A067E1K2-F1-model_v6.pdb')),
 ('AF-A0A075B634-F1-model_v6',
  PosixPath('../data/predictions/train/AF-A0A075B634-F1-model_v6.pdb'),
  PosixPath('../data/pdb/train/AF-A0A075B634-F1-model_v6.pdb')),
 ('AF-A0A075B653-F1-model_v6',
  PosixPath('../data/predictions/train/AF-A0A075B653-F1-model_v6.pdb'),
  PosixPath('../data/pdb/train/AF-A0A075B653-F1-model_v6.pdb')),
 ('AF-A0A075B658-F1-model_v6',
  PosixPath('../data/predictions/train/AF-A0A075B658-F1-model_v6.pdb'),
  PosixPath('../data/pdb/train/AF-A0A075B658-F1-model_v6.pdb')),
 ('AF-A0A077YUX2-F1-model_v6',
  PosixPath('../data/predictions/train/AF-A0A077YUX2-F1-model_v6.pdb'),
  PosixPath('../data/pdb/train/AF-A0A077YUX2-F1-model_v6.